In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import sys
import re

projectRoot = os.path.abspath(os.path.join(os.getcwd(), '..'))
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)
from helpers import initDB

In [ ]:
teamLinks2026 = [
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-a/eda3d93a',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-ar/3bdfef5c',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-d/d9d4e46a',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-dc/ebc7e070',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-dr/1804b11c',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-psc/1f6f71b5',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-arw/20222e5a',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-wa/7fc0dab0',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2026/teams/university-int-b/90acdd56'
]

<div class='alert alert-block alert-info'>
<b>Team Games</b>
</div>

In [ ]:
from graphQlQueries import teamFixtureQuery

def directTeamRequest(teamURL, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    teamID = teamURL.rstrip('/').split('/')[-1]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": teamURL,
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "teamFixture",
        "variables": {"teamID": teamID},
        "query": teamFixtureQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()

teamPage = directTeamRequest(teamLinks2026[1])

In [ ]:
def parseTeam(teamPage):
    data = teamPage
    team = re.findall(r'\((.*?)\)', data['discoverTeam']['name'])[0]
    rounds = data['discoverTeamFixture']
    summaries = []
    for round in rounds:
        summary= []
        summary.append(round['name'])
        status = round['fixture']
        if status['byes'] != []:
            summary.extend(['BYE', np.nan, np.nan, np.nan, np.nan, np.nan]) #home team, away team, time, date, venue, venueURL
        else:
            details = status['games'][0]
            summary.append(details['home']['name'].split('(')[0].strip()) #home team
            summary.append(details['away']['name'].split('(')[0].strip()) #away team
            summary.append(details['allocation']['time']) #time
            summary.append(details['allocation']['dateTimeList'][0]['date']) #date
            summary.append(details['allocation']['court']['name']) #venue
            latitude = details['allocation']['court']['latitude']
            longitude = details['allocation']['court']['longitude']
            summary.append(f'https://www.google.com/maps?q={latitude},{longitude}') #venue url
        summary.append(team) #uni team
        summary.append(round['isStale']) #round complete
        summaries.append(summary)
    df = pd.DataFrame(summaries, columns=['Round', 'HomeTeam', 'AwayTeam', 'Time', 'Date', 'Venue', 'VenueURL', 'UniTeam', 'RoundComplete'])
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    missingDates = df['Date'].isna()
    df.loc[missingDates, 'Date'] = df['Date'].ffill()[missingDates] + pd.Timedelta(days=7)
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time
    df['Central'] = np.nan
    df['Boundary'] = np.nan
    df['Goal'] = np.nan
    teamMap = {'A': 'A', 'AR': 'AR', 'ARW': 'ARW', 'DC':'GUL', 'D': 'PIR', 'DR': 'DIN', 'PSC': 'PSC', 'WA': 'WA', 'INT-B': 'INT'}
    df['UniTeam'] = df['UniTeam'].map(teamMap)
    df['GameID'] = df['UniTeam'] + ':' + df['Round'] + ':26' 
    df = df[['Round', 'HomeTeam', 'AwayTeam', 'Venue', 'VenueURL', 'Date', 'Time', 'GameID', 'UniTeam', 'Central', 'Boundary', 'Goal', 'RoundComplete']].copy()
    return df

parseTeam(teamPage)

<div class='alert alert-block alert-info'>
<b>Club Season Games</b>
</div>

In [ ]:
def clubFixtures():
    engine = initDB()
    teams = []
    for team in teamLinks2026:
        teamPage = directTeamRequest(team)
        teamDF = parseTeam(teamPage)
        teams.append(teamDF)
    clubDF = pd.concat(teams, ignore_index=True)
    clubDF.to_sql('seasonfixtures', con=engine, if_exists='replace', index=False)
    return clubDF

clubFixtures()